# NETRA - Neural Edge Tracking for Road Anomalies
## FASE 1: Environment & Project Setup

**Kompetisi:** FIND IT - Track A: The Edge Vision  
**Tim:** Sepupu Stanley - Universitas Gadjah Mada  
**Tanggal:** 10 April 2026  
**Deadline:** 12 April 2026

---

### Tujuan Fase 1
- Verifikasi environment Python yang reproducible
- Clone repository YOLOv5 (v7.0)
- Install semua dependency yang dibutuhkan
- Download MiDaS Small v2.1 ONNX (~50 MB)
- Membangun struktur direktori proyek NETRA
- Sanity check: pastikan semua modul dapat di-import tanpa error

---

### Definition of Done
- [ ] `python -c "import torch, onnxruntime, cv2"` berjalan tanpa error
- [ ] Struktur direktori `/kaggle/working/{data,models,depth,scripts,outputs}` sudah ada
- [ ] File `midas_small.onnx` sudah ada di `/kaggle/working/depth/` dan ukurannya <= 50 MB
- [ ] `requirements.txt` proyek sudah berisi semua dependency dengan versi terkunci

## Task 1.1 - Verifikasi Python Environment

Di Kaggle, virtual environment sudah terisolasi secara otomatis. Task ini memverifikasi versi Python dan memastikan kita menggunakan Python 3.9+ sesuai spesifikasi PRD. Kaggle saat ini menggunakan Python 3.12 yang sepenuhnya kompatibel.

In [1]:
import sys
import platform
import os

print("=" * 60)
print("NETRA - Fase 1: Environment & Project Setup")
print("=" * 60)
print(f"Python version  : {sys.version}")
print(f"Platform        : {platform.platform()}")
print(f"Working dir     : {os.getcwd()}")
print(f"Kaggle input    : /kaggle/input")
print(f"Kaggle working  : /kaggle/working")

# Validasi versi Python (3.9+ didukung, termasuk 3.12 yang digunakan Kaggle)
major, minor = sys.version_info[:2]
if major == 3 and minor >= 9:
    print(f"\n[OK] Python {major}.{minor} - versi didukung (3.9+)")
elif major == 3 and minor < 9:
    print(f"\n[WARN] Python {major}.{minor} - disarankan upgrade ke Python 3.9+")
else:
    raise RuntimeError(f"Python {major}.{minor} tidak didukung! Butuh Python 3.x (x >= 9)")

NETRA - Fase 1: Environment & Project Setup
Python version  : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform        : Linux-6.6.113+-x86_64-with-glibc2.35
Working dir     : /kaggle/working
Kaggle input    : /kaggle/input
Kaggle working  : /kaggle/working

[OK] Python 3.12 - versi didukung (3.9+)


## Task 1.2 - Clone Repository YOLOv5 (v7.0)

Clone repository resmi Ultralytics YOLOv5 dan checkout ke versi stabil v7.0 sesuai PRD. Versi ini dipilih karena stabilitas dan kompatibilitas dengan constraint kompetisi.

In [2]:
import os
import subprocess

YOLOV5_DIR = "/kaggle/working/yolov5"

if os.path.exists(YOLOV5_DIR):
    print(f"[SKIP] YOLOv5 sudah ada di {YOLOV5_DIR}")
else:
    print("[INFO] Cloning YOLOv5 repository...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "v7.0",
         "https://github.com/ultralytics/yolov5", YOLOV5_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"[ERROR] Clone gagal:\n{result.stderr}")
        raise RuntimeError("Git clone YOLOv5 v7.0 gagal")
    print(result.stdout)
    print(result.stderr)

# Verifikasi checkout v7.0
result = subprocess.run(
    ["git", "-C", YOLOV5_DIR, "describe", "--tags"],
    capture_output=True, text=True
)
print(f"[OK] YOLOv5 version tag: {result.stdout.strip()}")

# Tampilkan struktur folder YOLOv5
top_items = os.listdir(YOLOV5_DIR)
print(f"\n[INFO] Isi folder YOLOv5: {sorted(top_items)}")

[INFO] Cloning YOLOv5 repository...

Cloning into '/kaggle/working/yolov5'...
Note: switching to '915bbf294bb74c859f0b41f1c23bc395014ea679'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false


[OK] YOLOv5 version tag: v7.0

[INFO] Isi folder YOLOv5: ['.dockerignore', '.git', '.gitattributes', '.github', '.gitignore', '.pre-commit-config.yaml', 'CONTRIBUTING.md', 'LICENSE', 'README.md', 'benchmarks.py', 'classify', 'data', 'detect.py', 'export.py', 'hubconf.py', 'models', 'requirements.txt', 'segment', 'setup.cfg', 'train.py', 't

## Task 1.3 - Install Semua Dependencies

Install dependency dari `requirements.txt` YOLOv5 dan tambahan library yang dibutuhkan NETRA:
- `onnxruntime` - runtime inferensi ONNX (constraint kompetisi: CPU-only)
- `onnx` - untuk export dan validasi model ONNX
- `opencv-python` - frame processing dan video I/O
- `tensorboard` - monitoring training
- `matplotlib`, `pillow`, `pandas` - visualisasi dan data processing

In [3]:
# Install dependency YOLOv5
print("[INFO] Installing YOLOv5 requirements...")
!pip install -q -r /kaggle/working/yolov5/requirements.txt
print("[OK] YOLOv5 requirements installed")

[INFO] Installing YOLOv5 requirements...
[OK] YOLOv5 requirements installed


In [4]:
# Install dependency tambahan NETRA
print("[INFO] Installing NETRA additional dependencies...")
NETRA_DEPS = [
    "onnxruntime>=1.16.0",       # CPU-optimized ONNX inference (constraint kompetisi)
    "onnx>=1.14.0",              # ONNX model export & validation
    "opencv-python-headless",    # OpenCV tanpa GUI (cocok untuk server/Kaggle)
    "tensorboard",               # Training monitoring
    "matplotlib",                # Visualisasi
    "Pillow",                    # Image I/O
    "pandas",                    # Data processing
    "PyYAML",                    # YAML config (data.yaml)
    "tqdm",                      # Progress bar
    "scipy",                     # Scientific computing (untuk evaluasi depth)
]
!pip install -q {' '.join(NETRA_DEPS)}
print("[OK] NETRA additional dependencies installed")

[INFO] Installing NETRA additional dependencies...
[OK] NETRA additional dependencies installed


In [5]:
# Freeze requirements dan simpan ke file
import subprocess

result = subprocess.run(
    ["pip", "freeze"],
    capture_output=True, text=True
)
req_path = "/kaggle/working/requirements.txt"
with open(req_path, "w") as f:
    f.write(result.stdout)

lines = result.stdout.strip().split("\n")
print(f"[OK] requirements.txt disimpan ke {req_path}")
print(f"[INFO] Total packages: {len(lines)}")

# Tampilkan dependency kunci NETRA
KEY_PACKAGES = ["torch", "onnxruntime", "onnx", "opencv", "tensorboard", "numpy", "scipy"]
print("\n[INFO] Versi dependency kunci NETRA:")
for line in lines:
    pkg_lower = line.lower()
    if any(k in pkg_lower for k in KEY_PACKAGES):
        print(f"  {line}")

[OK] requirements.txt disimpan ke /kaggle/working/requirements.txt
[INFO] Total packages: 873

[INFO] Versi dependency kunci NETRA:
  numpy==2.0.2
  onnx==1.20.1
  onnxruntime==1.24.4
  opencv-contrib-python==4.13.0.92
  opencv-python==4.13.0.92
  opencv-python-headless==4.13.0.92
  pytorch-ignite==0.5.3
  pytorch-lightning==2.6.1
  scipy==1.16.3
  tensorboard==2.19.0
  tensorboard-data-server==0.7.2
  torch @ https://download.pytorch.org/whl/cpu/torch-2.10.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
  torchao==0.10.0
  torchaudio @ https://download.pytorch.org/whl/cpu/torchaudio-2.10.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl
  torchcodec @ https://download.pytorch.org/whl/cpu/torchcodec-0.10.0-cp312-cp312-manylinux_2_28_x86_64.whl
  torchdata==0.11.0
  torchinfo==1.8.0
  torchmetrics==1.9.0
  torchsummary==1.5.1
  torchtune==0.6.1
  torchvision @ https://download.pytorch.org/whl/cpu/torchvision-0.25.0%2Bcpu-cp312-cp312-manylinux_2_28_x86_64.whl


## Task 1.4 - Download MiDaS Small v2.1 ONNX

Download model MiDaS Small v2.1 dalam format ONNX dari repository Intel ISL MiDaS. Model ini digunakan untuk monocular depth estimation - komponen utama kedua NETRA setelah YOLOv5n-p6.

**Spesifikasi model (hasil inspeksi aktual):**
- Format: ONNX (sesuai constraint framework kompetisi)
- Ukuran target: ~50 MB (akan dicek setelah download)
- Input: RGB image resize ke **256x256** (bukan 384x384 — dikonfirmasi dari model shape)
- Output: Disparity map shape [1, 256, 256] (inverse depth, relatif)

In [6]:
import urllib.request
import os

DEPTH_DIR = "/kaggle/working/depth"
MIDAS_PATH = os.path.join(DEPTH_DIR, "midas_small.onnx")

# URL MiDaS Small v2.1 dari Intel ISL MiDaS GitHub releases
MIDAS_URL = "https://github.com/isl-org/MiDaS/releases/download/v2_1/model-small.onnx"

os.makedirs(DEPTH_DIR, exist_ok=True)

if os.path.exists(MIDAS_PATH):
    size_mb = os.path.getsize(MIDAS_PATH) / (1024 * 1024)
    print(f"[SKIP] MiDaS sudah ada: {MIDAS_PATH} ({size_mb:.1f} MB)")
else:
    print(f"[INFO] Downloading MiDaS Small v2.1 ONNX...")
    print(f"       URL: {MIDAS_URL}")

    def reporthook(block_num, block_size, total_size):
        downloaded = block_num * block_size
        total_mb = total_size / (1024 * 1024)
        done_mb = min(downloaded, total_size) / (1024 * 1024)
        pct = min(done_mb / total_mb * 100, 100)
        print(f"\r  [{pct:5.1f}%] {done_mb:.1f} / {total_mb:.1f} MB", end="", flush=True)

    urllib.request.urlretrieve(MIDAS_URL, MIDAS_PATH, reporthook)
    print()  # newline setelah progress

# Verifikasi ukuran file
size_mb = os.path.getsize(MIDAS_PATH) / (1024 * 1024)
print(f"\n[INFO] Ukuran file: {size_mb:.2f} MB")

if size_mb <= 50:
    print(f"[OK]  Ukuran {size_mb:.2f} MB <= 50 MB - memenuhi constraint kompetisi")
else:
    print(f"[WARN] Ukuran {size_mb:.2f} MB > 50 MB - perlu INT8 quantization di Fase 5")

[INFO] Downloading MiDaS Small v2.1 ONNX...
       URL: https://github.com/isl-org/MiDaS/releases/download/v2_1/model-small.onnx
  [100.0%] 63.7 / 63.7 MB

[INFO] Ukuran file: 63.67 MB
[WARN] Ukuran 63.67 MB > 50 MB - perlu INT8 quantization di Fase 5


In [7]:
# Task 1.4b - INT8 Quantization jika MiDaS > 50 MB
# Sesuai PRD: "Gunakan quantisasi INT8 jika perlu" (Section 3.1, Constraint C1)
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

DEPTH_DIR      = "/kaggle/working/depth"
MIDAS_PATH     = os.path.join(DEPTH_DIR, "midas_small.onnx")
MIDAS_Q_PATH   = os.path.join(DEPTH_DIR, "midas_small_int8.onnx")
SIZE_LIMIT_MB  = 50

original_mb = os.path.getsize(MIDAS_PATH) / (1024 * 1024)
print(f"[INFO] Ukuran model original : {original_mb:.2f} MB")

if original_mb <= SIZE_LIMIT_MB:
    # Sudah memenuhi constraint, tidak perlu quantisasi
    MIDAS_PATH = MIDAS_PATH
    print(f"[OK]  Model sudah <= {SIZE_LIMIT_MB} MB, quantisasi tidak diperlukan")
else:
    print(f"[WARN] Model {original_mb:.2f} MB > {SIZE_LIMIT_MB} MB — menerapkan INT8 dynamic quantization...")

    if os.path.exists(MIDAS_Q_PATH):
        print(f"[SKIP] Quantized model sudah ada: {MIDAS_Q_PATH}")
    else:
        quantize_dynamic(
            model_input  = MIDAS_PATH,
            model_output = MIDAS_Q_PATH,
            weight_type  = QuantType.QInt8
        )
        print("[OK]  Quantization selesai")

    quantized_mb = os.path.getsize(MIDAS_Q_PATH) / (1024 * 1024)
    print(f"\n[INFO] Ukuran setelah INT8 quantization : {quantized_mb:.2f} MB")
    print(f"[INFO] Reduksi ukuran                   : {original_mb - quantized_mb:.2f} MB ({(1 - quantized_mb/original_mb)*100:.1f}% lebih kecil)")

    if quantized_mb <= SIZE_LIMIT_MB:
        print(f"[OK]  {quantized_mb:.2f} MB <= {SIZE_LIMIT_MB} MB — constraint kompetisi TERPENUHI")
        MIDAS_PATH = MIDAS_Q_PATH   # Gunakan model quantized mulai sekarang
    else:
        raise RuntimeError(f"Setelah quantisasi masih {quantized_mb:.2f} MB > {SIZE_LIMIT_MB} MB. Coba kurangi resolusi input.")

print(f"\n[INFO] MIDAS_PATH aktif: {MIDAS_PATH}")

[INFO] Ukuran model original : 63.67 MB
[WARN] Model 63.67 MB > 50 MB — menerapkan INT8 dynamic quantization...
[OK]  Quantization selesai

[INFO] Ukuran setelah INT8 quantization : 16.53 MB
[INFO] Reduksi ukuran                   : 47.14 MB (74.0% lebih kecil)
[OK]  16.53 MB <= 50 MB — constraint kompetisi TERPENUHI

[INFO] MIDAS_PATH aktif: /kaggle/working/depth/midas_small_int8.onnx


In [8]:
# Validasi model ONNX bisa di-load oleh ONNX Runtime
import onnxruntime as ort
import numpy as np

print("[INFO] Memvalidasi MiDaS ONNX dengan ONNX Runtime...")

sess_options = ort.SessionOptions()
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# Load model - CPU only (sesuai constraint kompetisi)
session = ort.InferenceSession(
    MIDAS_PATH,
    sess_options=sess_options,
    providers=["CPUExecutionProvider"]
)

# Tampilkan info model & deteksi input size secara otomatis
print("\n[INFO] MiDaS Model Info:")
inp_meta = session.get_inputs()[0]
out_meta = session.get_outputs()[0]
print(f"  Input  - name: {inp_meta.name}, shape: {inp_meta.shape}, dtype: {inp_meta.type}")
print(f"  Output - name: {out_meta.name}, shape: {out_meta.shape}, dtype: {out_meta.type}")

# Deteksi input size dari model secara dinamis (agar tidak hardcode)
INPUT_H = inp_meta.shape[2]  # index 2 = height
INPUT_W = inp_meta.shape[3]  # index 3 = width
print(f"\n[INFO] MiDaS input size terdeteksi: {INPUT_H}x{INPUT_W}")

# Simpan konstanta untuk digunakan di fase berikutnya
MIDAS_INPUT_SIZE = (INPUT_H, INPUT_W)
print(f"[INFO] MIDAS_INPUT_SIZE = {MIDAS_INPUT_SIZE} (akan digunakan di Fase 4)")

# Test inferensi dengan dummy input sesuai shape model yang sebenarnya
input_name = inp_meta.name
dummy_input = np.random.rand(1, 3, INPUT_H, INPUT_W).astype(np.float32)
output = session.run(None, {input_name: dummy_input})

print(f"\n[OK] Dummy inference berhasil! Output shape: {output[0].shape}")
print(f"     Output range: [{output[0].min():.4f}, {output[0].max():.4f}]")
print("[OK] MiDaS Small ONNX siap digunakan")

[INFO] Memvalidasi MiDaS ONNX dengan ONNX Runtime...

[INFO] MiDaS Model Info:
  Input  - name: 0, shape: [1, 3, 256, 256], dtype: tensor(float)
  Output - name: 797, shape: [1, 256, 256], dtype: tensor(float)

[INFO] MiDaS input size terdeteksi: 256x256
[INFO] MIDAS_INPUT_SIZE = (256, 256) (akan digunakan di Fase 4)

[OK] Dummy inference berhasil! Output shape: (1, 256, 256)
     Output range: [650.6024, 1214.6945]
[OK] MiDaS Small ONNX siap digunakan


## Task 1.5 - Membangun Struktur Direktori Proyek

Membuat struktur direktori standar NETRA di `/kaggle/working/`. Struktur ini akan digunakan di semua fase berikutnya:

```
/kaggle/working/
|-- data/           <- Dataset setelah preprocessing (Fase 2)
|-- models/         <- Model weights (.pt, .onnx)
|-- depth/          <- MiDaS ONNX + calibration params
|-- scripts/        <- Script utility (benchmark, calibrate, demo)
|-- outputs/        <- Hasil inferensi, demo images, evaluasi
|   |-- demo/
|-- yolov5/         <- Repository YOLOv5 v7.0
|-- requirements.txt
|-- README.md
```

In [9]:
import os

BASE_DIR = "/kaggle/working"

# Definisi struktur direktori NETRA
DIRS = [
    "data",
    "models",
    "depth",
    "scripts",
    "outputs/demo",
]

print("[INFO] Membuat struktur direktori NETRA...")
for d in DIRS:
    full_path = os.path.join(BASE_DIR, d)
    os.makedirs(full_path, exist_ok=True)
    print(f"  [OK] {full_path}")

# Buat file .gitkeep agar folder tidak kosong
for d in DIRS:
    gitkeep = os.path.join(BASE_DIR, d, ".gitkeep")
    if not os.path.exists(gitkeep):
        open(gitkeep, 'w').close()

print("\n[OK] Semua direktori berhasil dibuat")

[INFO] Membuat struktur direktori NETRA...
  [OK] /kaggle/working/data
  [OK] /kaggle/working/models
  [OK] /kaggle/working/depth
  [OK] /kaggle/working/scripts
  [OK] /kaggle/working/outputs/demo

[OK] Semua direktori berhasil dibuat


In [10]:
# Buat README.md proyek
README_CONTENT = """# NETRA - Neural Edge Tracking for Road Anomalies

Sistem deteksi dini kerusakan jalan berbasis Edge Computer Vision.

**Kompetisi:** FIND IT - Track A: The Edge Vision
**Tim:** Sepupu Stanley - Universitas Gadjah Mada

## Arsitektur Pipeline
1. YOLOv5n-p6 - Deteksi lubang jalan (bounding box)
2. MiDaS Small v2.1 - Estimasi kedalaman monocular
3. Point Adjustment - Konversi depth relatif ke metrik (meter)
4. Pinhole Camera Model - Estimasi luas area lubang (m2)
5. Severity Classifier - DARURAT / SEDANG / PEMELIHARAAN RUTIN

## Struktur Direktori
- data/           : Dataset split (train/val/test)
- models/         : Model weights (.pt, .onnx)
- depth/          : MiDaS ONNX + calibration_params.json
- scripts/        : Script utility
- outputs/        : Hasil inferensi & demo
- yolov5/         : YOLOv5 v7.0 repository
- requirements.txt

## Cara Menjalankan Inferensi
python inference.py --image path/to/image.jpg

## Constraint Kompetisi
- Ukuran model : <= 50 MB
- Platform     : CPU-only (Intel Core i5 Gen 8)
- Latency      : <= 3 detik per frame
- Framework    : ONNX Runtime
- Mode         : Offline total

## Referensi
- Wang et al. (2024) - IEEE ICSIDP 2024. DOI: 10.1109/ICSIDP62679.2024.10869122
- YOLOv5 v7.0: https://github.com/ultralytics/yolov5
- MiDaS: https://github.com/isl-org/MiDaS
"""

readme_path = os.path.join(BASE_DIR, "README.md")
with open(readme_path, "w") as f:
    f.write(README_CONTENT)
print(f"[OK] README.md dibuat di {readme_path}")

[OK] README.md dibuat di /kaggle/working/README.md


## Task 1.6 - Sanity Check Environment

Verifikasi final bahwa semua modul kritis dapat di-import dan berjalan tanpa error. Ini adalah **gate check** sebelum melanjutkan ke Fase 2.

In [11]:
import importlib

# Daftar modul yang wajib tersedia
REQUIRED_MODULES = [
    ("torch",           "PyTorch - training YOLOv5"),
    ("onnxruntime",     "ONNX Runtime - CPU inferensi (constraint kompetisi)"),
    ("onnx",            "ONNX - model export & validasi"),
    ("cv2",             "OpenCV - frame processing & video I/O"),
    ("numpy",           "NumPy - matrix operations"),
    ("PIL",             "Pillow - image I/O"),
    ("pandas",          "Pandas - dataset management"),
    ("yaml",            "PyYAML - konfigurasi data.yaml"),
    ("matplotlib",      "Matplotlib - visualisasi"),
    ("tqdm",            "tqdm - progress bar"),
    ("scipy",           "SciPy - evaluasi depth estimation"),
    ("tensorboard",     "TensorBoard - training monitoring"),
]

print("=" * 60)
print("SANITY CHECK - Verifikasi Semua Module")
print("=" * 60)

all_ok = True
for module_name, description in REQUIRED_MODULES:
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", "(versi tidak tersedia)")
        print(f"  [OK] {module_name:<20} v{version:<15} - {description}")
    except ImportError as e:
        print(f"  [FAIL] {module_name:<18} - GAGAL: {e}")
        all_ok = False

print("=" * 60)
if all_ok:
    print("[PASS] Semua modul berhasil di-import - Fase 1 SELESAI")
else:
    print("[FAIL] Ada modul yang gagal - periksa instalasi di atas")

SANITY CHECK - Verifikasi Semua Module
  [OK] torch                v2.10.0+cpu      - PyTorch - training YOLOv5
  [OK] onnxruntime          v1.24.4          - ONNX Runtime - CPU inferensi (constraint kompetisi)
  [OK] onnx                 v1.20.1          - ONNX - model export & validasi
  [OK] cv2                  v4.13.0          - OpenCV - frame processing & video I/O
  [OK] numpy                v2.0.2           - NumPy - matrix operations
  [OK] PIL                  v11.3.0          - Pillow - image I/O
  [OK] pandas               v2.3.3           - Pandas - dataset management
  [OK] yaml                 v6.0.3           - PyYAML - konfigurasi data.yaml
  [OK] matplotlib           v3.10.0          - Matplotlib - visualisasi
  [OK] tqdm                 v4.67.3          - tqdm - progress bar
  [OK] scipy                v1.16.3          - SciPy - evaluasi depth estimation
  [OK] tensorboard          v2.19.0          - TensorBoard - training monitoring
[PASS] Semua modul berhasil di-im

In [12]:
# Verifikasi YOLOv5 dapat diakses sebagai modul
import sys
YOLOV5_DIR = "/kaggle/working/yolov5"
if YOLOV5_DIR not in sys.path:
    sys.path.insert(0, YOLOV5_DIR)

try:
    from models.common import DetectMultiBackend
    from utils.general import non_max_suppression
    from utils.torch_utils import select_device
    print("[OK] YOLOv5 modules dapat di-import (DetectMultiBackend, NMS, select_device)")
except ImportError as e:
    print(f"[FAIL] YOLOv5 import gagal: {e}")

[OK] YOLOv5 modules dapat di-import (DetectMultiBackend, NMS, select_device)


## Task 1.7 - Verifikasi GPU (Opsional)

Cek ketersediaan GPU untuk mempercepat training di Fase 3. GPU digunakan **hanya saat training** - deployment dan benchmark menggunakan CPU-only sesuai constraint kompetisi.

In [13]:
import torch

print("=" * 60)
print("Task 1.7 - GPU / Hardware Check")
print("=" * 60)

# PyTorch & CUDA info
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU             : {gpu_name}")
    print(f"VRAM            : {vram_gb:.1f} GB")
    print(f"CUDA version    : {torch.version.cuda}")
    print("\n[OK] GPU tersedia - Training Fase 3 akan berjalan lebih cepat")
    print("     (CPU-only tetap digunakan untuk inferensi/benchmark)")
else:
    print("\n[WARN] GPU tidak tersedia - aktifkan GPU di Settings Kaggle:")
    print("       Settings -> Accelerator -> GPU T4 x2")
    print("       Training Fase 3 akan sangat lambat tanpa GPU!")

# ONNX Runtime providers
import onnxruntime as ort
print(f"\nONNX Runtime version    : {ort.__version__}")
print(f"Available providers     : {ort.get_available_providers()}")
print(f"Deployment provider     : CPUExecutionProvider (constraint kompetisi)")

Task 1.7 - GPU / Hardware Check
PyTorch version : 2.10.0+cpu
CUDA available  : False

[WARN] GPU tidak tersedia - aktifkan GPU di Settings Kaggle:
       Settings -> Accelerator -> GPU T4 x2
       Training Fase 3 akan sangat lambat tanpa GPU!

ONNX Runtime version    : 1.24.4
Available providers     : ['AzureExecutionProvider', 'CPUExecutionProvider']
Deployment provider     : CPUExecutionProvider (constraint kompetisi)


In [14]:
import os

BASE_DIR        = "/kaggle/working"
MIDAS_ORIG      = os.path.join(BASE_DIR, "depth", "midas_small.onnx")
MIDAS_INT8      = os.path.join(BASE_DIR, "depth", "midas_small_int8.onnx")
SIZE_LIMIT_MB   = 50

print("=" * 60)
print("RINGKASAN - Fase 1: Definition of Done")
print("=" * 60)

checks = []

# Check 1: Modul kritis dapat di-import
try:
    import torch, onnxruntime, cv2
    checks.append((True, "import torch, onnxruntime, cv2 - OK"))
except ImportError as e:
    checks.append((False, f"import gagal: {e}"))

# Check 2: Direktori proyek ada
required_dirs = ["data", "models", "depth", "scripts", "outputs"]
missing_dirs = [d for d in required_dirs
                if not os.path.isdir(os.path.join(BASE_DIR, d))]
if not missing_dirs:
    checks.append((True, f"Direktori proyek lengkap: {required_dirs}"))
else:
    checks.append((False, f"Direktori hilang: {missing_dirs}"))

# Check 3: MiDaS ONNX tersedia dan <= 50 MB
# Prioritaskan model INT8 quantized, fallback ke original
if os.path.exists(MIDAS_INT8):
    active_model = MIDAS_INT8
    label = "midas_small_int8.onnx (quantized)"
elif os.path.exists(MIDAS_ORIG):
    active_model = MIDAS_ORIG
    label = "midas_small.onnx (original)"
else:
    active_model = None
    label = "tidak ditemukan"

if active_model:
    size_mb = os.path.getsize(active_model) / (1024 * 1024)
    if size_mb <= SIZE_LIMIT_MB:
        checks.append((True, f"{label} ({size_mb:.2f} MB <= {SIZE_LIMIT_MB} MB) - constraint TERPENUHI"))
    else:
        checks.append((False, f"{label} ({size_mb:.2f} MB > {SIZE_LIMIT_MB} MB) - quantisasi gagal"))
else:
    checks.append((False, "MiDaS ONNX TIDAK ditemukan di /kaggle/working/depth/"))

# Check 4: requirements.txt tersedia
req_path = os.path.join(BASE_DIR, "requirements.txt")
if os.path.exists(req_path):
    with open(req_path) as f:
        pkg_count = len(f.readlines())
    checks.append((True, f"requirements.txt ada ({pkg_count} packages)"))
else:
    checks.append((False, "requirements.txt TIDAK ada"))

# Check 5: YOLOv5 v7.0 tersedia
yolov5_dir = os.path.join(BASE_DIR, "yolov5")
if os.path.exists(yolov5_dir) and os.path.exists(os.path.join(yolov5_dir, "train.py")):
    checks.append((True, "YOLOv5 v7.0 repository tersedia"))
else:
    checks.append((False, "YOLOv5 repository TIDAK lengkap"))

# Print hasil
passed = 0
for ok, msg in checks:
    status = "[PASS]" if ok else "[FAIL]"
    print(f"  {status} {msg}")
    if ok:
        passed += 1

print("=" * 60)
total = len(checks)
print(f"Hasil: {passed}/{total} checks lulus")
if passed == total:
    print("\n[SUCCESS] FASE 1 SELESAI!")
    print("          Lanjut ke FASE 2 - Data Preparation & Preprocessing")
else:
    print(f"\n[INCOMPLETE] {total - passed} check gagal - perbaiki sebelum ke Fase 2")

RINGKASAN - Fase 1: Definition of Done
  [PASS] import torch, onnxruntime, cv2 - OK
  [PASS] Direktori proyek lengkap: ['data', 'models', 'depth', 'scripts', 'outputs']
  [PASS] midas_small_int8.onnx (quantized) (16.53 MB <= 50 MB) - constraint TERPENUHI
  [PASS] requirements.txt ada (873 packages)
  [PASS] YOLOv5 v7.0 repository tersedia
Hasil: 5/5 checks lulus

[SUCCESS] FASE 1 SELESAI!
          Lanjut ke FASE 2 - Data Preparation & Preprocessing
